# Python から Julia への翻訳メモ
- strip() は

## データ ("input.txt") 読み込み
```docs = [l.strip() for l in open('input.txt').read().strip().split('\n') if l.strip()] # list[str] of documents```

1. open('input.txt') → ファイルを開く
2. .read() → ファイル全体を1つの文字列として読み込む
3. .strip() → 先頭・末尾の空白・改行を除去
4. .split('\n') → 改行で分割してリスト（行ごと）にする
5. for l in ... → 各行を l としてループ
6. if l.strip() → 空白除去後に空でない行だけフィルタ（空行を除外）
7. l.strip() → 各行の前後の空白を除去して最終結果に

ということで、 strip がまず重要だけど、空白を除くということ。
一度ファイル全体を文字列として空白を除くのに使ってから、
行ごとに strip もしている。その結果として単語（名前）が入る

Julia では docs は基本的に Vector{Strings} だが Vector{Substring{String}} となっているのに注意が必要、かも

In [ ]:
## tokenizer 作成


## n_step = 10_000 としたら

step 9999 / 10000 | loss 3.3113
step 10000 / 10000 | loss 3.2053

--- inference (new, hallucinated names) ---
sample  1: yxuifypzwoxwhikv
sample  2: xhzgxpeff
sample  3: xefpzbq
sample  4: rzlbubrxjwnhwbmm
sample  5: zrhzmhogvjnqchdd
sample  6: bpxyqcg
sample  7: ppjrhbxmdeokirnd
sample  8: hpeixtuvxrkkjgjr
sample  9: jfqsrtedfbrxgfty
sample 10: kxearwutksidzldy
sample 11: kmcbscbuagipzzzg
sample 12: cmqnndhzoxibdtch
sample 13: yglrmhmj
sample 14: rqegqafjodpqdrjm
sample 15: farlxcnowluzbphw
sample 16: sfsjtifcndtcd
sample 17: sjcwxsfzjnqnupxe
sample 18: cyxhy
sample 19: egxkeevaz
sample 20: efrpcbegjxrllfll

となって、過学習？　loss が大きいし

これは抜けがあって、もう直させた
```
Python 版の対応箇所：


def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._children:
            build_topo(child)
        topo.append(v)  # ← これが欠けていた
topo にノードが追加されないと reverse(topo) のループが空になり、勾配が全く計算されないため、パラメータが更新されずランダムのままになります。```



## BOS について

`BOS は、始点と終点の両方を担うので、 BOS ではなく EOS でも良いってこと？`

はい、その通りです。このコードでは1つの特殊トークンが文の開始と終了の両方を兼ねています。


tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
`#          ↑ 開始                                      ↑ 終了`
名前を BOS にしても EOS にしても SPECIAL にしても動作は同じです。要は「文の境界」を示す唯一の特殊トークンがあれば十分で、名前は慣習の問題です。

Karpathy が BOS を選んだ理由はおそらく、推論時の使い方が BOS 的だからです：


token_id = BOS          # BOS から生成を開始
for pos_id in range(block_size):
    ...
    if token_id == BOS:  # 再び BOS が出たら終了
        break
「BOS で始めて、BOS が出たら止める」と読めるので、開始トークンとして命名する方が自然というだけです。


## micropokemon_katakana.jl を作成しました。

英語版との違い：

英語版	カタカナ版
データソース	PokeAPI	sindresorhus/pokemon の ja.json
キャッシュ	pokemon_names.txt	pokemon_names_ja.txt
vocab size	~30 (a-z + ハイフン等)	~80 (カタカナ + ー等)
n_embd	16	32 (vocab が大きいので増加)
vocab が約3倍なので n_embd を 16→32 に増やしています。それ以外のモデル構造は同じです。


julia micropokemon_katakana.jl          # 1000ステップ
julia micropokemon_katakana.jl 5000     # 5000ステップ

# GPT-2 からの変更点

## GeLU (Gaussian Error Linear Unit)

ReLU は `max(0, x)` で x < 0 をバッサリ切り捨てますが、GeLU は**確率的に**ゲートします：

```
GeLU(x) = x * Φ(x)
```

`Φ(x)` は標準正規分布の累積分布関数（0〜1の値）です。

```
x = -2  →  Φ(-2) ≈ 0.02  →  GeLU ≈ -2 * 0.02 = -0.04  （ほぼ遮断）
x =  0  →  Φ(0)  = 0.5   →  GeLU =  0 * 0.5  =  0
x =  2  →  Φ(2)  ≈ 0.98  →  GeLU ≈  2 * 0.98 =  1.96  （ほぼ通過）
```

直感的には：
- **ReLU**: x < 0 なら完全に0、x > 0 ならそのまま（硬い境界）
- **GeLU**: x が大きいほど通しやすく、小さいほど通しにくい（滑らかな境界）

GPT-2 が GeLU を採用し、以降の LLM で標準になりました。microgpt は簡略化のため ReLU を使っています。

---

## LayerNorm

各層の出力を**平均0、分散1**に正規化します：

```
LayerNorm(x) = γ * (x - μ) / √(σ² + ε) + β
```

- `μ` = ベクトル x の平均
- `σ²` = ベクトル x の分散
- `γ, β` = 学習可能なパラメータ（スケールとシフト）

microgpt が代わりに使っている **RMSNorm** との比較：

```
RMSNorm(x) = x / √(mean(x²) + ε)
```

| | LayerNorm | RMSNorm |
|---|---|---|
| 平均を引く | はい | **いいえ** |
| 学習パラメータ γ, β | あり | **なし**（microgptでは） |
| 計算コスト | やや重い | 軽い |

RMSNorm は「平均を引く」ステップと学習パラメータを省略した簡易版です。LLaMA 等の最近のモデルでも RMSNorm が使われており、実用上 LayerNorm と同等の性能が出ることがわかっています。

---

要するに microgpt は GPT-2 の構造を、教育目的で最小限に簡略化しています：
- **GeLU → ReLU**: 滑らかなゲート → 単純な閾値
- **LayerNorm → RMSNorm**: 平均引き+学習パラメータ → 二乗平均で割るだけ

# 改変してみること
- ポケモンとかにはしてみたので、自分の馴染みのあるデータでやってみる
- proto-prompt として、名前を途中まで与えて、そこからどういう名前が補完されて出てきやすいかを見てみるようにする